In [0]:
%python
base_path = "/Volumes/workspace/bronze/landing/AdventureWorksCSV-main/"
for f in dbutils.fs.ls(base_path):
    print(f.name)

In [0]:
%python
base_path = "/Volumes/workspace/bronze/landing/AdventureWorksCSV-main/AdventureWorksCSV-main/"
for f in dbutils.fs.ls(base_path):
    print(f.name)

In [0]:
%python
from pyspark.sql.functions import current_timestamp, lit

base_path = "/Volumes/workspace/bronze/landing/AdventureWorksCSV-main/AdventureWorksCSV-main/"

files = {
    "sales_order_header": "Sales SalesOrderHeader.csv",
    "sales_order_detail": "Sales SalesOrderDetail.csv",
    "customer": "Sales Customer.csv",
    "sales_territory": "Sales SalesTerritory.csv",
    "product": "Production Product.csv",
    "product_category": "Production ProductCategory.csv",
    "address": "Person Address.csv",
}

for table_name, filename in files.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", False)   # Bronze: keep everything as string, no type guessing
          .csv(base_path + filename))
    df = (df.withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", lit(filename)))
    df.write.mode("overwrite").saveAsTable(f"workspace.bronze.{table_name}")
    print(f"Loaded {table_name}: {df.count()} rows")

In [0]:
%sql
SELECT 'address' AS tbl, COUNT(*) AS n FROM workspace.bronze.address
UNION ALL SELECT 'customer', COUNT(*) FROM workspace.bronze.customer
UNION ALL SELECT 'product', COUNT(*) FROM workspace.bronze.product
UNION ALL SELECT 'product_category', COUNT(*) FROM workspace.bronze.product_category
UNION ALL SELECT 'sales_order_detail', COUNT(*) FROM workspace.bronze.sales_order_detail
UNION ALL SELECT 'sales_order_header', COUNT(*) FROM workspace.bronze.sales_order_header
UNION ALL SELECT 'sales_territory', COUNT(*) FROM workspace.bronze.sales_territory;